# tp-14 戦略探索 レベル 1: オッズを使う戦略（購入時点のオッズの雑音つき）＋購入額の段階づけ

**目的**: 『期待値が高い買い目だけを買う』系の戦略を、パラメータの格子で網羅して試し、**多重検定を補正したうえで**回収率 100% 超が本当にあるかを調べる。購入額の段階づけ（500 / 1000 / 2000 円、見送り）は基本の部品として、全戦略に適用して比べる。

## 設定
- 買い目の表は tp-09 の取消馬修正後の版（モデル確率とオッズ、確定の払戻つき）。判断オッズ = **確定オッズ × 対数正規の雑音**（σ = 0.1 / 0.2。購入時点と確定のずれの代用）。精算は確定の払戻。σ = 0（確定オッズを知っている先読み）は上限の参考として別に出し、検定には入れない。
- 買い目の選び方: 券種、期待値 p × オッズ ≥ τ（1.0 / 1.1 / 1.25 / 1.5）、確率 p ≥ p_min（0 / 0.05 / 0.15 / 0.3）、期待値の高い順に最大 k 点（1 / 3）、確率の出どころ（市場とのブレンド / モデル単独）。
- 購入額: 毎回 1000 円（flat）／ 期待値で 500・1000・2000 円（ev）／ 期待値に的中確率の下限（15% 未満は見送り、30% 未満は 500 円、45% 未満は 1000 円まで）を加えたもの（evhit）。
- **多重検定**: 試した全戦略に White の reality check をかけ、最良が偶然でない（p 値）かを見る。選択は valid 2022〜2024 だけ。上位 5 戦略を test 2025 で一度だけ評価。

In [ ]:
import os, sys
from pathlib import Path


def _find_root():
    p = Path.cwd().resolve()
    for q in [p, *p.parents]:
        if (q / "AGENTS.md").exists() and (q / "src").is_dir():
            return q
    raise RuntimeError("リポジトリルートが見つかりません")


REPO_ROOT = _find_root()
os.chdir(REPO_ROOT)
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))
from src.utils.project_env import load_project_dotenv
load_project_dotenv()

In [ ]:
import json, subprocess, time, itertools
from datetime import datetime
import numpy as np
import pandas as pd
import mlflow

from src.pipeline.betting import lab
from src.pipeline.mlflow.train_utils import require_tracking_server

BET = REPO_ROOT / "data/local/modeling/betting"
TAG = os.environ.get("TP14_TAG", "lab1_v1")
OUT = BET / "lab" / TAG
OUT.mkdir(parents=True, exist_ok=True)
LOCK = BET / "lab" / f"TEST2025_EVALUATED_{TAG}.json"
GIT_COMMIT = subprocess.run(["git", "rev-parse", "--short", "HEAD"], capture_output=True, text=True, cwd=REPO_ROOT).stdout.strip()
mlflow.set_experiment("keiba-tp-betting-lab")
require_tracking_server()
YEARS = ["2022", "2023", "2024", "2025"]
TD = BET / "tickets/cmp_v2_K12_v2"
assert json.loads((TD / "2022.json").read_text()).get("clean") == 1
RAW = {y: pd.read_parquet(TD / f"{y}.parquet") for y in YEARS}
RACES = {y: sorted(RAW[y]["race_id"].unique()) for y in YEARS}
print({y: len(v) for y, v in RACES.items()})
BETS = ["tansho", "fukusho", "umaren", "wide", "umatan"]
TAUS, PMINS, TOPKS, PSRCS = (1.0, 1.1, 1.25, 1.5), (0.0, 0.05, 0.15, 0.3), (1, 3), ("p_comb", "p_model")
SCHEMES = ("flat", "ev", "evhit")
SIGMAS = (0.1, 0.2, 0.0)

In [ ]:
def run_year(y, sigma):
    d = lab.prepare_tickets(RAW[y], sigma, seed=100 + int(y))
    out = {}
    for bet, tau, pm, k, ps in itertools.product(BETS, TAUS, PMINS, TOPKS, PSRCS):
        port = lab.select_portfolio(d, bet, tau, pm, k, ps)
        if port.empty:
            continue
        for sc in SCHEMES:
            out[(sigma, bet, tau, pm, k, ps, sc)] = lab.settle_portfolio(port, lab.tier_stake(port, sc))
    return out


t0 = time.time()
RES = {}
for sigma in SIGMAS:
    for y in YEARS[:3]:
        for key, v in run_year(y, sigma).items():
            RES[(key, y)] = v
    print(f"σ={sigma}: {time.time() - t0:.0f}s", flush=True)
KEYS = sorted({k for (k, y) in RES})
print("戦略数（σ・券種・τ・p_min・k・確率の出どころ・購入額の組合せ）:", len(KEYS))

In [ ]:
rows = []
for key in KEYS:
    parts = [RES[(key, y)] for y in YEARS[:3] if (key, y) in RES]
    d = pd.concat(parts)
    st, rt = d["stake"].sum(), d["ret"].sum()
    if len(d) < 300 or st <= 0:
        continue
    per_year = {y: (RES[(key, y)]["ret"].sum() / RES[(key, y)]["stake"].sum()) if (key, y) in RES and RES[(key, y)]["stake"].sum() > 0 else np.nan for y in YEARS[:3]}
    rows.append({"σ": key[0], "券種": key[1], "τ": key[2], "p_min": key[3], "k": key[4], "確率": key[5], "購入額": key[6], "レース数": len(d), "賭け金(円)": st, "利益(円)": rt - st, "回収率": rt / st,
                 "正の年数": int(sum(v > 1 for v in per_year.values() if v == v)), **{f"roi_{y}": v for y, v in per_year.items()}})
S = pd.DataFrame(rows)
S.to_csv(OUT / "valid_all_strategies.csv", index=False)
print(len(S), "戦略（レース数 300 以上）")
R = S[S["σ"] > 0]
print("現実的な雑音あり（σ>0）の上位 12（回収率）:")
display(R.sort_values("回収率", ascending=False).head(12).round(3))
print("購入額の方式別（σ>0・全戦略）の回収率・利益の中央値と、正の利益だった戦略の数:")
display(R.groupby("購入額").agg(戦略数=("回収率", "size"), 回収率の中央値=("回収率", "median"), 利益の中央値=("利益(円)", "median"), 正の利益=("利益(円)", lambda s: int((s > 0).sum()))).round(3))
print("σ=0（確定オッズを知っている先読み、参考）の上位 5:")
display(S[S["σ"] == 0].sort_values("回収率", ascending=False).head(5).round(3))

## 多重検定（reality check、σ > 0 の全戦略）

In [ ]:
all_r = sorted(r for y in YEARS[:3] for r in RACES[y]); ix = {r: i for i, r in enumerate(all_r)}
keys_rc = [tuple(r[["σ", "券種", "τ", "p_min", "k", "確率", "購入額"]]) for _, r in R.iterrows()]
M = np.zeros((len(all_r), len(keys_rc)), dtype="float32")
for j, key in enumerate(keys_rc):
    for y in YEARS[:3]:
        v = RES.get((key, y))
        if v is not None and len(v):
            M[[ix[r] for r in v["race_id"]], j] = ((v["ret"] - v["stake"]) / 1000.0).to_numpy()
rc = lab.reality_check(M.astype("float64"), n_boot=500)
best = keys_rc[rc["best"]]
print(f"試した戦略 {rc['n_strategies']} 個。標準化した最良 = {best}、平均損益 {rc['best_mean'] * 1000:.1f} 円/レース、reality check の p 値 = {rc['p_value']:.3f}")
print("平均損益が正の戦略数:", int((M.mean(0) > 0).sum()), "/", M.shape[1])
json.dump({"n_strategies": rc["n_strategies"], "best": [str(x) for x in best], "best_mean_yen_per_race": rc["best_mean"] * 1000, "p_value": rc["p_value"], "n_positive": int((M.mean(0) > 0).sum())}, open(OUT / "reality_check_valid.json", "w"), ensure_ascii=False, indent=1)

## test 2025（一度だけ）
valid で回収率が高い上位 5 戦略（σ > 0、レース数 500 以上）を 2025 年に適用。雑音は 2025 年用に新しく引く。

In [ ]:
if LOCK.exists() and os.environ.get("TP14_FORCE") != "1":
    raise SystemExit(f"test 2025 は評価済みです（{LOCK}）")
cand = R[R["レース数"] >= 500].sort_values("回収率", ascending=False).head(5)
rows, PM = [], []
ix25 = {r: i for i, r in enumerate(RACES["2025"])}
for _, c in cand.iterrows():
    sg = c["σ"]
    d = lab.prepare_tickets(RAW["2025"], sg, seed=100 + 2025)
    port = lab.select_portfolio(d, c["券種"], c["τ"], c["p_min"], int(c["k"]), c["確率"])
    v = lab.settle_portfolio(port, lab.tier_stake(port, c["購入額"]))
    ci = lab.roi_ci(v.assign(), n_boot=1000)
    rows.append({"σ": sg, "券種": c["券種"], "τ": c["τ"], "p_min": c["p_min"], "k": c["k"], "確率": c["確率"], "購入額": c["購入額"], "valid の回収率": c["回収率"], "test 2025 の回収率": ci["roi"], "lo": ci["lo"], "hi": ci["hi"], "レース数": ci["races"],
                 "test 2025 の利益(円)": float(v["ret"].sum() - v["stake"].sum())})
    full = np.zeros(len(ix25)); full[[ix25[r] for r in v["race_id"]]] = ((v["ret"] - v["stake"]) / 1000.0).to_numpy(); PM.append(full)
T = pd.DataFrame(rows); display(T.round(3)); T.to_csv(OUT / "test2025_top5.csv", index=False)
rc5 = lab.reality_check(np.column_stack(PM), n_boot=1000)
print("test 2025: 5 戦略の最良に対する reality check の p 値 =", round(rc5["p_value"], 3))
res = {"evaluated_at": datetime.now().isoformat(timespec="seconds"), "tag": TAG, "git_commit": GIT_COMMIT, "n_strategies": int(len(S)), "valid_reality_check": json.load(open(OUT / "reality_check_valid.json")), "test2025": T.to_dict("records"), "test2025_p": rc5["p_value"]}
(OUT / "report.json").write_text(json.dumps(res, ensure_ascii=False, indent=1, default=str), encoding="utf-8")
with mlflow.start_run(run_name=f"tp-14-lab1-{TAG}", tags={"notebook": "tp-14", "git_commit": GIT_COMMIT, "test_year": "2025"}):
    mlflow.log_metrics({"n_strategies": float(len(S)), "valid_rc_p": float(res["valid_reality_check"]["p_value"]), "test2025_rc_p": float(rc5["p_value"])})
    for i_, r_ in T.iterrows():
        mlflow.log_metrics({f"test2025_roi__{i_:02d}": float(r_["test 2025 の回収率"])})
    for p in OUT.glob("*"):
        mlflow.log_artifact(str(p), "lab1")
LOCK.write_text(json.dumps({"evaluated_at": res["evaluated_at"], "git_commit": GIT_COMMIT, "note": "test 2025 のレベル1評価は実施済み"}, ensure_ascii=False))
print("完了")